<a href="https://colab.research.google.com/github/claramanolache/DeepLearning/blob/main/Stock_Market.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install opendatasets
!pip install legacy-cgi

from practice notebook (added opendatasets for kaggle import)

In [2]:
import copy

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils import data

import opendatasets as od
import pandas as pds

In [3]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

#Importing Dataset

In [4]:
od.download(
    "https://www.kaggle.com/datasets/zahrashahzahi/stock-market-dataset/data")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: claramanolache
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/zahrashahzahi/stock-market-dataset


100%|██████████| 2.08M/2.08M [00:00<00:00, 155MB/s]

In [5]:
import os

folder = "stock-market-dataset"

print(os.listdir(folder))

['Market.csv']


In [6]:
file =('stock-market-dataset/Market.csv')
df = pds.read_csv(file)

In [7]:
df.shape
df.head()

,Index,Date,Open,High,Low,Close,Adj Close,Volume
0,NYA,12/31/1965,528.690002,528.690002,528.690002,528.690002,528.690002,0.0
1,NYA,1/3/1966,527.210022,527.210022,527.210022,527.210022,527.210022,0.0
2,NYA,1/4/1966,527.840027,527.840027,527.840027,527.840027,527.840027,0.0
3,NYA,1/5/1966,531.119995,531.119995,531.119995,531.119995,531.119995,0.0
4,NYA,1/6/1966,532.070007,532.070007,532.070007,532.070007,532.070007,0.0


In [13]:
print(df.shape)

(112457, 8)


#Dataset Visuals

In [8]:
df['Index'].unique()

array(['NYA', 'IXIC', 'HSI', '000001.SS', 'GSPTSE', '399001.SZ', 'NSEI',
       'GDAXI', 'KS11', 'SSMI', 'TWII', 'J203.JO', 'N225', 'N100'],
      dtype=object)

In [9]:
count = (df['Index'] == 'NYA').sum()
print(count)

13948


# Hyperparameters

experimenting with window size

In [10]:
window_size_1 = 15
window_size_2 = 30
window_size_3 = 100

In [ ]:
model = nn.Sequential(
    nn.RNN(
        input_size=8,
        hidden_size=32,
        num_layers=1,
        batch_first=True
    ),
    nn.Linear(32, 1)
)

In [ ]:
model_1 = nn.Sequential( #TODO
    nn.RNN(
        input_size=8,
        hidden_size=32,
        num_layers=1,
        batch_first=True
    ),
    nn.Linear(32, 1)
)

In [ ]:
n_epochs = 10
learning_rate = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.MSELoss()
patience = 5

#Seperating X and Y

In [21]:
df_train = df[:90000]
df_val = df[90000:102000]
df_test = df[102000:]

In [22]:
X_train, y_train = df_train.drop(columns={'Close','Adj Close'}, axis=1), df['Close']
X_val, y_val = df_val.drop(columns={'Close','Adj Close'}, axis=1), df['Close']
X_test, y_test = df_test.drop(columns={'Close','Adj Close'}, axis=1), df['Close']
X_test

,Index,Date,Open,High,Low,Volume
102000,N225,8/14/2001,11587.040040,11936.750000,11587.040040,0.0
102001,N225,8/15/2001,11812.000000,11823.919920,11648.860350,0.0
102002,N225,8/16/2001,11645.299810,11645.299810,11450.769530,0.0
102003,N225,8/17/2001,11537.690430,11581.250000,11412.360350,0.0
102004,N225,8/20/2001,11347.240230,11358.089840,11239.450200,0.0
...,...,...,...,...,...,...
112452,N100,5/27/2021,1241.119995,1251.910034,1241.119995,379696400.0
112453,N100,5/28/2021,1249.469971,1259.209961,1249.030029,160773400.0
112454,N100,5/31/2021,1256.079956,1258.880005,1248.140015,91173700.0
112455,N100,6/1/2021,1254.609985,1265.660034,1254.609985,155179900.0


#Setting up models

code from the practice notebook

In [14]:
def create_data_loader(X, y, batch_size=32, shuffle=True):
    return data.DataLoader(
        data.TensorDataset(
            torch.FloatTensor(X),
            torch.FloatTensor(y)
        ),
        batch_size=batch_size,
        shuffle=shuffle
    )

In [ ]:
train_loader = create_data_loader(X_train_win, y_train_win)
val_loader = create_data_loader(X_val_win, y_val_win)
test_loader = create_data_loader(X_test_win, y_test_win, shuffle=False)

len(train_loader), len(val_loader), len(test_loader)

In [ ]:
def get_next_prediction(
    model: nn.Sequential, inputs: torch.Tensor
) -> torch.Tensor:
    outputs, _ = model[0](inputs)
    last_time_step = outputs[:, -1, :]
    return model[1](last_time_step)

In [ ]:
model.to(device)
model.train()
for epoch in range(n_epochs):
    total_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = get_next_prediction(model, X_batch)

        loss = criterion(y_pred, y_batch)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

In [ ]:
avg_costs = []
best_cost = float("inf")
best_params = None

model.to(device)
for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        y_pred = get_next_prediction(model, X_batch)

        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            y_pred = get_next_prediction(model, X_batch)

            loss = criterion(y_pred, y_batch)
            val_loss += loss.item()
    avg_costs.append(val_loss / len(val_loader))

    print(f"Epoch {epoch:3d} | Validation Cost: {avg_costs[-1]:.4f} | Best: {best_cost:.4f}")
    if avg_costs[-1] < best_cost:
        best_cost = avg_costs[-1]
        best_params = copy.deepcopy(model.state_dict())
    elif len(avg_costs) > patience and min(avg_costs[-patience:]) > best_cost:
        print("Early Stopping Triggered")
        break